In [ ]:
!pip install -q -U transformers chromadb sentence-transformers "pillow<12.0"

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)
print("Hugging Face login successful!")

import torch
import time
from transformers import AutoProcessor, AutoModelForImageTextToText
from sentence_transformers import SentenceTransformer
import chromadb
from PIL import Image
import os
import shutil

INPUT_DATA_PATH = "/kaggle/input/knowledge-dataset/"

READ_ONLY_DB_PATH = os.path.join(INPUT_DATA_PATH, "embedding_vectors")
IMG_PATH = os.path.join(INPUT_DATA_PATH, "all_images")

WRITABLE_DB_PATH = "/kaggle/working/my_vector_db"

print(f"Copying database from {READ_ONLY_DB_PATH} to {WRITABLE_DB_PATH}...")

if os.path.exists(WRITABLE_DB_PATH):
    print("Writable copy already exists. Removing old one.")
    shutil.rmtree(WRITABLE_DB_PATH)

shutil.copytree(READ_ONLY_DB_PATH, WRITABLE_DB_PATH)
print("Database copied successfully.")

print(f"Loading images : {IMG_PATH}")

print("Loading Gemma 3 VLM...")
device = "cuda" if torch.cuda.is_available() else "cpu"
gemma_processor = AutoProcessor.from_pretrained("google/gemma-3-4b-it")
gemma_model = AutoModelForImageTextToText.from_pretrained("google/gemma-3-4b-it",torch_dtype=torch.bfloat16).to(device)
print("Gemma 3 loaded.")

print("Loading text embedding model (google/embeddinggemma-300m) on CPU...")
text_embed_model = SentenceTransformer('google/embeddinggemma-300m', device="cpu")
print("Embedding model loaded.")

print("Loading vector database...")
client = chromadb.PersistentClient(path=WRITABLE_DB_PATH)
collection = client.get_collection(name="assam_disasters_multimodal")
print(f"Database loaded. Total items: {collection.count()}")

def analyze_image(image_path, analysis_prompt="Describe this image in detail."):
    print(f"ANALYZING IMAGE: {image_path}")

    try:
        image = Image.open(image_path).convert("RGB")
    except FileNotFoundError:
        print(f"ERROR: Image file not found at {image_path}. Please check the path.")
        return
    except Exception as e:
        print(f"ERROR loading image: {e}")
        return


    messages = [
        {
            "role": "user","content": [{"type": "image", "url": image_path},{"type": "text", "text": analysis_prompt}]
        }
    ]
    print("Generating analysis with Gemma 3...")

    start_time = time.time()

    inputs = gemma_processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = gemma_model.generate(**inputs, max_new_tokens=512)

    start_index = inputs["input_ids"].shape[-1]
    answer = gemma_processor.decode(outputs[0][start_index:], skip_special_tokens=True)

    end_time = time.time()
    latency = end_time - start_time

    print(answer)
    print("===================================")
    print(f" Generation Latency: {latency:.2f} seconds")
    print("===================================\n")

def run_multimodal_rag(question, n_results=3):
    print(f"\n--- New Question ---")
    print(f"Q: {question}\n")

    retrieval_start = time.time()
    print("Embedding question and retrieving documents...")
    text_embed_model.to(device)
    query_embedding = text_embed_model.encode(question).tolist()
    text_embed_model.to("cpu")
    torch.cuda.empty_cache()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    retrieval_end = time.time()
    retrieval_latency = retrieval_end - retrieval_start

    if not results or not results['documents'][0]:
        print("We couldn't find any relevant documents in my knowledge base.")
        return
    context_text_list = []
    context_image_paths = []

    documents = results['documents'][0]
    metadatas = results['metadatas'][0]

    for doc, meta in zip(documents, metadatas):
        context_text_list.append(doc)
        image_paths_str = meta.get('image_paths', '')
        if image_paths_str:
            paths = image_paths_str.split(',')
            for p in paths:
                img_filename = os.path.basename(p)
                correct_path = os.path.join(IMG_PATH, img_filename)
                if os.path.exists(correct_path):
                    context_image_paths.append(correct_path)

    context_image_paths = sorted(list(set(context_image_paths)))

    print(f"Found {len(context_text_list)} text chunks and {len(context_image_paths)} unique images.")

    prompt = ("Question")

    for i, text in enumerate(context_text_list):
        prompt += f"Chunk {i+1}:\n{text}\n\n"

    prompt += "End"

    if context_image_paths:
        prompt += (
            f"You are also provided with {len(context_image_paths)} relevant "
            "images in the input. Use them as context if helpful.\n\n"
        )

    prompt += f"--- QUESTION ---\n{question}"

    messages = [{"role": "user", "content": []}]
    messages[0]["content"].append({"type": "text", "text": prompt})

    MAX_IMAGES = 2
    for img_path in context_image_paths[:MAX_IMAGES]:
        try:
            messages[0]["content"].append({"type": "image", "url": img_path})
        except Exception as e:
            print(f"Warning: could not load image {img_path}: {e}")

    print("Generating answer with Gemma 3...")

    gen_start = time.time()

    inputs = gemma_processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = gemma_model.generate(**inputs, max_new_tokens=512)

    start_index = inputs["input_ids"].shape[-1]
    answer = gemma_processor.decode(outputs[0][start_index:], skip_special_tokens=True)

    gen_end = time.time()
    gen_latency = gen_end - gen_start

    print("\n--- Answer ---")
    print(f"A: {answer}")

    print("\n--- Sources ---")
    for meta in metadatas:
        print(f"  - {meta['source_pdf']}, Page: {meta['page_number']}")
    print("\n--- Performance Metrics ---")
    print(f"Retrieval Latency:  {retrieval_latency:.2f}s")
    print(f"Generation Latency: {gen_latency:.2f}s")
    print(f"Total Turnaround:   {retrieval_latency + gen_latency:.2f}s")

def run_analysis():
    print("Gemma 3 Analysis Mode Selector")

    while True:
        choice = input("Select mode (T: Text RAG | I: Image Analysis | Q: Quit): ").upper().strip()

        if choice == 'Q':
            print("Exit")
            break

        elif choice == 'T':
            question = input("Enter your RAG query: ")
            n_results = 3
            run_multimodal_rag(question, n_results)

        elif choice == 'I':
            print("Image Analysis")

            image_path = input("Enter the FULL path to the image file")
            analysis_prompt = input("Enter the analysis prompt: ")

            analyze_image(image_path, analysis_prompt)

        else:
            print("Invalid choice. Please enter T, I, or Q.")

run_analysis()